Séance 1
Informations	Détails
Nom	EDOUN MARC
Porteur	P1
Étapes assignées	1.1 et 1.8


# Carnet de démarrage — sujet 03 · Home Credit

**Objectif du sujet** : décider d'accorder ou de refuser un crédit.

Ce carnet est un **châssis**, pas une solution. Il fait quatre choses et s'arrête :

1. il **trouve** vos fichiers, quelle que soit la plateforme ;
2. il **charge** la table efficacement — cache Parquet construit une fois, projection de colonnes ;
3. il **profile** les données : dimensions, types, cible, manquants ;
4. il **écrit** une soumission au bon format, avec une valeur de référence volontairement nulle.

**Ce qu'il ne fait pas** : choisir votre métrique, votre découpage de validation, votre modèle, ni vos
variables. Ces décisions sont le sujet, et c'est ce que la grille note.

8,07 % de défauts. Les manquants ne sont pas comblés : c'est une information.

---

## Où mettre ce carnet

| Plateforme | Ce qu'il faut faire |
|---|---|
| **Le dossier du sujet** | **https://drive.google.com/drive/folders/17v1qEMO8FohlDPU99fAjEcuy_nl-Nim5?usp=sharing** — téléchargez-le avant tout (le lien figure aussi dans l'énoncé) |
| **En local (VS Code, Jupyter)** | placez le carnet **dans le dossier téléchargé**, ou définissez `DONNEES_PROJET` vers ce dossier |
| **Google Colab** | déposez les fichiers dans `MyDrive/Sujet_03_Home_Credit/` — le carnet les trouvera ; ou téléversez-les dans `/content/` |
| **Kaggle** | créez un jeu de données avec vos fichiers, **Add Input** — `/kaggle/input` est exploré automatiquement |

**Rien n'est codé en dur** : le carnet cherche, et s'il ne trouve pas, il **dit précisément** ce qu'il a
cherché et où.


## Les fichiers de ce sujet

| Fichier | Rôle |
|---|---|
| `train.csv` | entraînement ou annexe |
| `a_decider.csv` | à prédire — la cible est absente |
| `HomeCredit_columns_description.csv` | entraînement ou annexe |

**Format de sortie attendu** : `SK_ID_CURR,TARGET`.
**Colonne d'identifiant** : `SK_ID_CURR`.
**Fichier à produire** : `soumission.csv`, à la racine de votre dépôt.


## Sur les données lourdes — et pourquoi pas « raylib »

`raylib` est une bibliothèque **graphique** (fenêtres, dessin, jeux) : elle ne lit aucune table. Elle
n'a rien à faire ici. Les outils qui répondent au besoin sont :

| Outil | Rôle | Présent par défaut ? |
|---|---|---|
| **Parquet + pyarrow** | format colonne compressé, lecture sélective | **oui** avec pandas récent |
| **Polars** | moteur colonne, `scan_csv` paresseux | non — `pip install polars` |
| **DuckDB** | SQL directement sur CSV/Parquet, sans chargement | non — `pip install duckdb` |

Ce carnet utilise **Parquet** (garanti), et le remplace par Polars ou DuckDB si vous les installez. Le
principe ne change pas : **ne jamais charger ce dont on n'a pas besoin**, et **payer la conversion
CSV → Parquet une seule fois**.


In [26]:
# --- 1. Réglages --------------------------------------------------------------
from pathlib import Path
import os
import pandas as pd

# Où télécharger vos données — le dossier de votre sujet sur le Drive du cours.
LIEN_DRIVE = "__LIEN__"

# ECHANTILLON : nombre de lignes lues pour démarrer. None = tout lire.
ECHANTILLON = 200_000
DOSSIER_CACHE = Path.cwd() / "_cache_parquet"


In [27]:
# --- 2. Trouver les données, sans rien coder en dur --------------------------
FICHIERS_ATTENDUS = ['train.csv', 'a_decider.csv', 'HomeCredit_columns_description.csv']
SUJET = "Sujet_03_Home_Credit"


def trouver_donnees(fichiers=FICHIERS_ATTENDUS, sujet=SUJET):
    """Cherche le dossier qui contient TOUS les fichiers attendus."""
    essais = []
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    ici = Path.cwd().resolve()
    pistes += [ici, ici / "donnees", ici / sujet]
    pistes += list(ici.parents)[:4]
    pistes += [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]

    for racine in pistes:
        if not racine.exists():
            continue
        candidates = [racine, racine / sujet]
        if racine.name in ("input", "MyDrive"):
            candidates += [d for d in racine.iterdir() if d.is_dir()]
        for c in candidates:
            essais.append(str(c))
            if all((c / f).is_file() for f in fichiers):
                return c
    raise FileNotFoundError(
        "Dossier de données introuvable.\n"
        "Il faut un dossier contenant : " + ", ".join(fichiers) + "\n"
        "Solutions : placez le carnet à côté de ce dossier, "
        "ou définissez DONNEES_PROJET=/chemin/vers/le/dossier.\n"
        "Chemins essayés : " + str(essais[:12])
    )


DOSSIER = trouver_donnees()
print("dossier de données :", DOSSIER)
print("fichiers vus :", sorted(p.name for p in DOSSIER.iterdir() if p.is_file())[:12])


dossier de données : /home/olalekan/D-p-t-de-groupe-6/edoun_marc
fichiers vus : ['HomeCredit_columns_description.csv', 'Sujet_03_Home_Credit_demarrage.ipynb', 'a_decider.csv', 'notebook.ipynb', 'soumission.csv', 'train.csv']


In [ ]:
# --- Inventoriser les tables  ----------------------------------------
import csv

for nom in FICHIERS_ATTENDUS:
    chemin = DOSSIER / nom
    with chemin.open("r", encoding="cp1252", newline="") as fichier:
        lecteur = csv.reader(fichier)
        nombre_colonnes = len(next(lecteur, []))
        nombre_lignes = sum(1 for _ in lecteur)

    print(f"{chemin.name} : {nombre_lignes} lignes, {nombre_colonnes} colonnes")


train.csv : 246008 lignes, 122 colonnes
a_decider.csv : 61503 lignes, 121 colonnes
HomeCredit_columns_description.csv : 219 lignes, 5 colonnes


In [29]:
# --- Indices de fuite de données par rapport à TARGET ------------------------
import pandas as pd

train_fuites = pd.read_csv(DOSSIER / "train.csv")
test_ids = pd.read_csv(DOSSIER / "a_decider.csv", usecols=["SK_ID_CURR"])["SK_ID_CURR"]
y = train_fuites["TARGET"]
variables = train_fuites.drop(columns=["TARGET", "SK_ID_CURR"], errors="ignore")
indices_fuite = []

for nom, serie in variables.items():
    paires = serie.notna() & y.notna()
    valeurs, cible = serie[paires], y[paires]

    if len(valeurs) and (valeurs.eq(cible).all() or valeurs.eq(1 - cible).all()):
        indices_fuite.append(f"{nom} : identique ou inverse de TARGET")
        continue

    if pd.api.types.is_numeric_dtype(serie):
        correlation = serie.corr(y)
        if pd.notna(correlation) and abs(correlation) >= 0.98:
            indices_fuite.append(f"{nom} : corrélation avec TARGET = {correlation:.3f}")
    elif serie.nunique(dropna=True) <= 20:
        tableau = pd.crosstab(serie, y)
        effectifs = tableau.sum(axis=1)
        groupes = tableau.loc[effectifs >= 50]
        couverture = groupes.to_numpy().sum() / max(1, paires.sum())
        purete = groupes.gt(0).sum(axis=1).le(1).all()
        if len(groupes) >= 2 and couverture >= 0.95 and purete:
            indices_fuite.append(f"{nom} : catégories presque parfaitement associées à TARGET")

mots_suspects = ("TARGET", "DEFAULT", "DELINQU", "OVERDUE", "PAYMENT_STATUS", "LOAN_STATUS")
for nom in variables.columns:
    if any(mot in nom.upper() for mot in mots_suspects):
        indices_fuite.append(f"{nom} : nom de colonne à vérifier")

chevauchement_ids = train_fuites["SK_ID_CURR"].isin(test_ids).sum()
ids_dupliques_train = train_fuites["SK_ID_CURR"].duplicated().sum()
ids_dupliques_test = test_ids.duplicated().sum()

print("Indices de fuite à vérifier :")
print("\n".join(indices_fuite) if indices_fuite else "Aucun indice évident détecté.")
print(f"IDs présents à la fois dans train et a_decider : {chevauchement_ids}")
print(f"IDs dupliqués — train : {ids_dupliques_train}, a_decider : {ids_dupliques_test}")
print("Une absence d'indice statistique ne garantit pas l'absence de fuite : vérifier la date et l'origine de chaque variable.")


Indices de fuite à vérifier :
Aucun indice évident détecté.
IDs présents à la fois dans train et a_decider : 0
IDs dupliqués — train : 0, a_decider : 0
Une absence d'indice statistique ne garantit pas l'absence de fuite : vérifier la date et l'origine de chaque variable.


In [30]:
# --- Entraînement sans fuite entre train et validation -----------------------
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

donnees = pd.read_csv(DOSSIER / "train.csv")
X = donnees.drop(columns=["TARGET", "SK_ID_CURR"])
y = donnees["TARGET"]
ids = donnees["SK_ID_CURR"]

# Découper avant de calculer les statistiques ou de transformer les variables.
X_train, X_val, y_train, y_val, ids_train, ids_val = train_test_split(
    X, y, ids, test_size=0.2, random_state=42, stratify=y
)

colonnes_numeriques = X_train.select_dtypes(include="number").columns
colonnes_categorielles = X_train.select_dtypes(exclude="number").columns
pretraitement = ColumnTransformer([
    ("numeriques", Pipeline([
        ("imputation", SimpleImputer(strategy="median")),
        ("normalisation", StandardScaler()),
    ]), colonnes_numeriques),
    ("categories", Pipeline([
        ("imputation", SimpleImputer(strategy="constant", fill_value="manquant")),
        ("encodage", OneHotEncoder(handle_unknown="ignore")),
    ]), colonnes_categorielles),
])

modele = Pipeline([
    ("pretraitement", pretraitement),
    ("classification", LogisticRegression(max_iter=200)),
])

# fit ne voit que X_train et y_train; X_val sert uniquement à l'évaluation.
modele.fit(X_train, y_train)
probabilites_val = modele.predict_proba(X_val)[:, 1]

assert set(ids_train).isdisjoint(set(ids_val)), "Un même identifiant est dans train et validation."
print(f"Lignes entraînement : {len(X_train)} ; validation : {len(X_val)}")
print(f"IDs communs entre les deux partitions : {len(set(ids_train) & set(ids_val))}")
print(f"ROC AUC validation : {roc_auc_score(y_val, probabilites_val):.4f}")
print("Imputation et encodage appris uniquement sur le train via Pipeline.")


Lignes entraînement : 196806 ; validation : 49202
IDs communs entre les deux partitions : 0
ROC AUC validation : 0.7459
Imputation et encodage appris uniquement sur le train via Pipeline.


In [31]:
# --- 3. Chargement efficace (table lourde) ------------------------------------
# Le cache Parquet s'écrit à côté du carnet, jamais dans les données : sur Kaggle,
# /kaggle/input est en lecture seule.
import pyarrow as pa
import pyarrow.csv as pvc
import pyarrow.parquet as pq


def table(nom, colonnes=None, complet=False):
    """Ouvre un CSV en profitant d'un cache Parquet.

    nom      : nom du fichier dans le dossier de données
    colonnes : liste de colonnes à garder (None = toutes) — le levier principal
    morceaux : si un entier, rend un itérateur de DataFrames de cette taille
    """
    source = DOSSIER / nom
    if not source.is_file():
        raise FileNotFoundError(f"{nom} introuvable dans {DOSSIER}")

    # lecture d'un échantillon : pas de cache, on veut seulement voir
    if ECHANTILLON and not complet:
        lots, reste = [], ECHANTILLON
        for lot in pvc.open_csv(str(source), read_options=pvc.ReadOptions(block_size=1 << 25)):
            lots.append(lot)
            reste -= lot.num_rows
            if reste <= 0:
                break
        t = pa.Table.from_batches(lots)
        if colonnes:
            t = t.select([c for c in colonnes if c in t.column_names])
        return t.to_pandas()

    # cache Parquet, construit une fois
    cache = DOSSIER_CACHE / (Path(nom).stem + ".parquet")
    if not cache.is_file():
        DOSSIER_CACHE.mkdir(parents=True, exist_ok=True)
        print(f"construction du cache Parquet : {cache.name} (une seule fois)")
        ecrivain = None
        try:
            for lot in pvc.open_csv(str(source), read_options=pvc.ReadOptions(block_size=1 << 26)):
                if colonnes:
                    lot = lot.select([c for c in colonnes if c in lot.schema.names])
                if ecrivain is None:
                    ecrivain = pq.ParquetWriter(cache, lot.schema)
                ecrivain.write_batch(lot)
        finally:
            if ecrivain is not None:
                ecrivain.close()
    fichier = pq.ParquetFile(cache)
    return fichier.read(columns=colonnes).to_pandas()


def table_par_morceaux(nom, colonnes=None, morceaux=200_000):
    """Itère sur la table par paquets — pour les traitements qui ne tiennent pas en mémoire."""
    table(nom, colonnes=colonnes)          # garantit que le cache existe
    cache = DOSSIER_CACHE / (Path(nom).stem + ".parquet")
    for lot in pq.ParquetFile(cache).iter_batches(batch_size=morceaux):
        yield lot.to_pandas()


In [32]:
# --- 4. Charger et regarder ---------------------------------------------------
entrainement = table("train.csv")
print("dimensions :", entrainement.shape)
print("colonnes :", len(entrainement.columns))
entrainement.head()


dimensions : (246008, 122)
colonnes : 122


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,123551,0,Cash loans,M,N,Y,0,202500.0,1078200.0,31653.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,400429,0,Cash loans,F,N,Y,2,148500.0,544491.0,20133.0,...,0,0,0,0,0.0,0.0,0.0,0.0,2.0,0.0
2,106912,0,Cash loans,F,Y,N,2,67500.0,363190.5,24268.5,...,0,0,0,0,0.0,0.0,0.0,1.0,1.0,3.0
3,383991,0,Revolving loans,F,N,Y,1,144000.0,450000.0,22500.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
4,423763,0,Cash loans,F,N,Y,0,67500.0,675000.0,21775.5,...,0,0,0,0,0.0,0.0,1.0,0.0,0.0,0.0


In [33]:
# --- 5. Profil de la cible ----------------------------------------------------
CIBLE = "TARGET"
if CIBLE in entrainement.columns:
    serie = entrainement[CIBLE]
    print("type :", serie.dtype)
    print("manquants :", int(serie.isna().sum()))
    print(serie.describe())
    if serie.nunique() <= 20:
        print(serie.value_counts(normalize=True).sort_index())
else:
    print(f"la cible {CIBLE} est absente de cette table — normal pour la table à prédire")


type : int64
manquants : 0
count    246008.000000
mean          0.080729
std           0.272419
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: TARGET, dtype: float64
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [34]:
# --- 6. Soumission de RÉFÉRENCE (format seulement) ----------------------------
# Cette valeur n'a AUCUNE prétention prédictive : elle vérifie le format.
COLONNES_SORTIE = ['SK_ID_CURR', 'TARGET']


def ecrire_soumission(identifiants, valeurs=None, chemin="soumission.csv"):
    """Écrit le fichier attendu. `valeurs` peut être une colonne, ou une matrice de probabilités."""
    sortie = pd.DataFrame({"SK_ID_CURR": identifiants})
    if len(COLONNES_SORTIE) == 2:
        sortie["TARGET"] = valeurs
    else:
        n = len(COLONNES_SORTIE) - 1
        matrice = valeurs if valeurs is not None else [[1.0 / n] * n for _ in identifiants]
        for j, colonne in enumerate(COLONNES_SORTIE[1:]):
            sortie[colonne] = [ligne[j] for ligne in matrice]
    sortie.to_csv(chemin, index=False)
    print(f"écrit : {chemin} · {len(sortie):,} lignes · colonnes {list(sortie.columns)}")
    if len(COLONNES_SORTIE) > 2:
        print("somme par ligne (doit valoir 1) :", sortie.iloc[:, 1:].sum(axis=1).round(6).unique()[:3])
    return sortie


a_predire = table("a_decider.csv", colonnes=["SK_ID_CURR"], complet=True)
identifiants = a_predire["SK_ID_CURR"].drop_duplicates().to_numpy()
print("identifiants à prédire :", len(identifiants))
ecrire_soumission(identifiants)


identifiants à prédire : 61503
écrit : soumission.csv · 61,503 lignes · colonnes ['SK_ID_CURR', 'TARGET']


,SK_ID_CURR,TARGET
0,424613,None
1,260084,None
2,214262,None
3,357905,None
4,101573,None
...,...,...
61498,342357,None
61499,382938,None
61500,235122,None
61501,348296,None


In [36]:
# --- 7. Le travail commence ICI -----------------------------------------------
# À vous de décider, et de justifier :
#   - la métrique, et ce qu'elle coûte ;
#   - le découpage de validation ;
#   - les variables, leur encodage, leur imputation ;
#   - le modèle, son réglage, son seuil.
# Un exemple de pipeline sans fuite se trouve dans la cellule précédente.
print("Cellule de départ : remplacez ces commentaires par votre propre modélisation.")


Cellule de départ : remplacez ces commentaires par votre propre modélisation.
